# A04 mnist — scratch

Mã huấn luyện tách từ notebook gốc, giữ tham số/thuật toán. Mặc định `RUN_TRAINING=False`: Run All chỉ kiểm imports/path và xem số liệu đã lưu; không fit/TEST. Muốn thực nghiệm mới phải tự bật cờ; mọi output legacy path được chuyển vào một thư mục `notebooks/executed/artifacts/` mới, không ghi đè kết quả gốc.


In [ ]:
# Migration r2: select this assignment's original logical paths, preserving frozen source.
from pathlib import Path as _Path
import sys as _sys
_repo = next(p for p in (_Path.cwd(), *_Path.cwd().parents) if (p / 'ASSIGN_REORGANIZATION_EXECUTION_PLAN.md').exists())
_sys.path.insert(0, str(_repo / '_shared'))
from assignment_runtime import activate as _activate
_migration = _activate('A04', 'torch')
_migration


In [ ]:
RUN_TRAINING = False
if RUN_TRAINING:
    from assignment_runtime import enable_training_output_overlay
    enable_training_output_overlay()
from pathlib import Path
import json
import pandas as pd


# 1. Assignment Overview


Notebook này xây pipeline Assignment 04 cho đúng bài toán CNN image classification trên MNIST: ảnh chữ số viết tay 28x28 grayscale, 10 classes. Notebook tập trung vào representation ảnh, convolution operations, NumPy CNN from scratch, Keras CNN, PyTorch CNN, so sánh framework và phân tích lỗi dự đoán.

In [ ]:
if RUN_TRAINING:
    from pathlib import Path
    import json
    import pandas as pd

    def find_project_root(start=None):
        start = Path.cwd() if start is None else Path(start)
        for path in [start.resolve(), *start.resolve().parents]:
            if (path / "datasets").exists() and (path / "pipeline04").exists():
                return path
        raise FileNotFoundError("Cannot locate project root.")

    PROJECT_ROOT = find_project_root()
    DATA_DIR = PROJECT_ROOT / "datasets" / "mnist"
    ARTIFACT_DIR = PROJECT_ROOT / "pipeline04" / "mnist"
    MODEL_DIR = ARTIFACT_DIR / "models"
    METRIC_DIR = ARTIFACT_DIR / "metrics"
    FIGURE_DIR = ARTIFACT_DIR / "figures"
    METADATA_DIR = ARTIFACT_DIR / "metadata"
    for directory in [DATA_DIR, MODEL_DIR, METRIC_DIR, FIGURE_DIR, METADATA_DIR]:
        directory.mkdir(parents=True, exist_ok=True)

    overview = {
        "Dataset": "MNIST",
        "Source": "TensorFlow/Keras official MNIST archive",
        "Task": "Image classification",
        "Image representation": "28 x 28 grayscale",
        "Classes": 10,
        "Notebook role": "Style reference for CIFAR-10 CNN notebook",
    }
    pd.DataFrame([overview]).T.rename(columns={0: "Value"})


**Nhận xét kết quả.**

- Bảng overview xác nhận notebook dùng MNIST image data thật, không còn dùng tabular dataset.
- Các artifact của phase MNIST được tách dưới `pipeline04/mnist/`.

# 2. Environment Setup


Import thư viện, cố định seed, kiểm tra version TensorFlow/PyTorch và xác định device. Output chỉ dùng thông tin môi trường cần thiết, không in path cá nhân.

In [ ]:
import os
import random
import time
import warnings
import urllib.request
from datetime import datetime, timezone
import joblib
import matplotlib.pyplot as plt
import numpy as np
import sklearn
from IPython.display import display
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
warnings.filterwarnings('ignore', category=FutureWarning)
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
CLASS_NAMES = [str(i) for i in range(10)]
env_info = {'Python executable': Path(os.sys.executable).name, 'NumPy': np.__version__, 'Pandas': pd.__version__, 'scikit-learn': sklearn.__version__, 'Seed': SEED}
pd.DataFrame([env_info]).T.rename(columns={0: 'Value'})


**Nhận xét kết quả.**

- Output cho biết notebook đang chạy bằng CPU hay GPU và version các framework.
- Seed được cố định để các split, initialization và DataLoader reproducible hơn.

# 3. Load Dataset


Load MNIST từ file cache trong `datasets/mnist/`. Nếu file chưa có, code có cơ chế tải lại từ TensorFlow/Keras official archive.

In [ ]:
if RUN_TRAINING:
    MNIST_URL = "https://storage.googleapis.com/tensorflow/tf-keras-datasets/mnist.npz"
    MNIST_PATH = DATA_DIR / "mnist.npz"

    if not MNIST_PATH.exists():
        urllib.request.urlretrieve(MNIST_URL, MNIST_PATH)

    with np.load(MNIST_PATH) as data:
        x_train_full = data["x_train"]
        y_train_full = data["y_train"]
        x_test_raw = data["x_test"]
        y_test = data["y_test"]

    load_summary = {
        "Cache file": MNIST_PATH.relative_to(PROJECT_ROOT).as_posix(),
        "Cache size MB": round(MNIST_PATH.stat().st_size / (1024 * 1024), 3),
        "x_train shape": x_train_full.shape,
        "y_train shape": y_train_full.shape,
        "x_test shape": x_test_raw.shape,
        "y_test shape": y_test.shape,
        "Image dtype": str(x_train_full.dtype),
        "Pixel min": int(x_train_full.min()),
        "Pixel max": int(x_train_full.max()),
        "Number of classes": int(np.unique(y_train_full).size),
    }
    pd.DataFrame([load_summary]).T.rename(columns={0: "Value"})


**Nhận xét kết quả.**

- MNIST cache nằm tại `datasets/mnist/mnist.npz`, kích thước khoảng 10.96 MB.
- Dataset load được 60,000 ảnh train gốc và 10,000 ảnh test; mỗi ảnh là grayscale 28x28 với 10 class.
- Pixel ban đầu ở thang 0-255, nên cần normalize trước khi train CNN.

# 4. Dataset Inspection


Kiểm tra phân phối class, hiển thị sample images và inspect một ảnh cụ thể để hiểu dữ liệu ảnh trước khi chuẩn hóa.

In [ ]:
if RUN_TRAINING:
    train_counts = np.bincount(y_train_full, minlength=10)
    test_counts = np.bincount(y_test, minlength=10)
    class_distribution = pd.DataFrame({
        "Class": np.arange(10),
        "TrainCount": train_counts,
        "TrainRate": train_counts / len(y_train_full),
        "TestCount": test_counts,
        "TestRate": test_counts / len(y_test),
    })

    sample_index = 0
    sample_info = {
        "Sample index": sample_index,
        "Shape": x_train_full[sample_index].shape,
        "Dtype": str(x_train_full[sample_index].dtype),
        "Min": int(x_train_full[sample_index].min()),
        "Max": int(x_train_full[sample_index].max()),
        "Label": int(y_train_full[sample_index]),
    }

    display(class_distribution.round(4))
    display(pd.DataFrame([sample_info]).T.rename(columns={0: "Value"}))

    fig, ax = plt.subplots(figsize=(6, 3.5))
    ax.bar(class_distribution["Class"].astype(str), class_distribution["TrainCount"], color="#4C78A8")
    ax.set_title("MNIST Train Class Distribution")
    ax.set_xlabel("Digit class")
    ax.set_ylabel("Count")
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / "class_distribution.png", dpi=160)
    plt.show()

    fig, axes = plt.subplots(2, 5, figsize=(8, 3.4))
    for digit, ax in enumerate(axes.ravel()):
        idx = np.where(y_train_full == digit)[0][0]
        ax.imshow(x_train_full[idx], cmap="gray")
        ax.set_title(f"Label {digit}")
        ax.axis("off")
    fig.suptitle("One MNIST Example per Class", y=1.03)
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / "one_example_per_class.png", dpi=160)
    plt.show()

    rng = np.random.default_rng(SEED)
    sample_ids = rng.choice(len(x_train_full), size=25, replace=False)
    fig, axes = plt.subplots(5, 5, figsize=(6, 6))
    for idx, ax in zip(sample_ids, axes.ravel()):
        ax.imshow(x_train_full[idx], cmap="gray")
        ax.set_title(str(int(y_train_full[idx])), fontsize=9)
        ax.axis("off")
    fig.suptitle("Random MNIST Samples", y=1.02)
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / "sample_grid.png", dpi=160)
    plt.show()


**Nhận xét kết quả.**

- Class distribution giúp kiểm tra MNIST có đủ 10 chữ số và không bị lệch class nghiêm trọng.
- Các grid ảnh cho thấy input là chữ số viết tay grayscale, đúng loại dữ liệu mà Conv2D có cơ sở dùng.

# 5. Data Preparation


Tạo train/validation/test split, normalize pixel và chuẩn bị hai layout tensor: Keras dùng channels-last, PyTorch dùng channels-first.

In [ ]:
if RUN_TRAINING:
    x_train_raw, x_valid_raw, y_train, y_valid = train_test_split(
        x_train_full,
        y_train_full,
        test_size=0.10,
        random_state=SEED,
        stratify=y_train_full,
    )

    X_train_keras = (x_train_raw.astype("float32") / 255.0)[..., np.newaxis]
    X_valid_keras = (x_valid_raw.astype("float32") / 255.0)[..., np.newaxis]
    X_test_keras = (x_test_raw.astype("float32") / 255.0)[..., np.newaxis]

    X_train_torch = np.transpose(X_train_keras, (0, 3, 1, 2))
    X_valid_torch = np.transpose(X_valid_keras, (0, 3, 1, 2))
    X_test_torch = np.transpose(X_test_keras, (0, 3, 1, 2))

    split_summary = pd.DataFrame([
        {"Split": "train", "Images": len(y_train), "Shape": X_train_keras.shape, "PixelMin": float(X_train_keras.min()), "PixelMax": float(X_train_keras.max())},
        {"Split": "validation", "Images": len(y_valid), "Shape": X_valid_keras.shape, "PixelMin": float(X_valid_keras.min()), "PixelMax": float(X_valid_keras.max())},
        {"Split": "test", "Images": len(y_test), "Shape": X_test_keras.shape, "PixelMin": float(X_test_keras.min()), "PixelMax": float(X_test_keras.max())},
    ])
    layout_summary = pd.DataFrame([
        {"Framework": "Keras/TensorFlow", "Layout": "B x H x W x C", "TrainShape": X_train_keras.shape},
        {"Framework": "PyTorch", "Layout": "B x C x H x W", "TrainShape": X_train_torch.shape},
    ])

    display(split_summary)
    display(layout_summary)


**Nhận xét kết quả.**

- Validation được lấy từ original training set: train 54,000, validation 6,000, test 10,000.
- Normalize `/255` không học tham số từ validation/test, nên không tạo leakage.
- Keras dùng tensor `(B, 28, 28, 1)`, còn PyTorch dùng `(B, 1, 28, 28)`.

# 6. Discover CNN Operations


Chạy các operation CNN cơ bản trên tensor nhỏ: Conv2D, ReLU, MaxPool, Flatten, Dense, logits và softmax.

**Nhận xét kết quả.**

- Conv2D giữ batch dimension và đổi channel từ 1 sang số filter.
- MaxPool giảm height/width, Flatten chuyển feature map thành vector, Dense tạo 10 logits cho 10 class.
- Softmax sum xấp xỉ 1, đúng tính chất phân phối xác suất.

# 7. Inspect CNN Building Blocks


Dùng một batch MNIST thật để kiểm tra shape qua từng building block trong cả Keras và PyTorch.

**Nhận xét kết quả.**

- Một batch có 8 ảnh tạo ra 8 hàng logits, mỗi hàng có 10 giá trị cho 10 chữ số.
- Keras dùng `(B, H, W, C)`, còn PyTorch dùng `(B, C, H, W)`.

# 8. CNN From Scratch with NumPy


Định nghĩa các layer NumPy có forward/backward thật: Conv2D, ReLU, MaxPool2D, Flatten, Dense, SoftmaxCrossEntropy và SGD update.

In [ ]:
if RUN_TRAINING:
    def im2col_nhwc(x, kh, kw, pad=0, stride=1):
        x_pad = np.pad(x, ((0, 0), (pad, pad), (pad, pad), (0, 0)), mode="constant")
        b, h, w, c = x_pad.shape
        out_h = (h - kh) // stride + 1
        out_w = (w - kw) // stride + 1
        cols = np.empty((x.shape[0], out_h, out_w, kh, kw, c), dtype=x.dtype)
        for i in range(kh):
            for j in range(kw):
                cols[:, :, :, i, j, :] = x_pad[:, i:i + out_h * stride:stride, j:j + out_w * stride:stride, :]
        return cols.reshape(x.shape[0] * out_h * out_w, kh * kw * c), x_pad.shape, out_h, out_w

    def col2im_nhwc(cols, x_shape, x_pad_shape, kh, kw, pad=0, stride=1):
        b, h, w, c = x_shape
        _, hp, wp, _ = x_pad_shape
        out_h = (hp - kh) // stride + 1
        out_w = (wp - kw) // stride + 1
        cols = cols.reshape(b, out_h, out_w, kh, kw, c)
        dx_pad = np.zeros(x_pad_shape, dtype=cols.dtype)
        for i in range(kh):
            for j in range(kw):
                dx_pad[:, i:i + out_h * stride:stride, j:j + out_w * stride:stride, :] += cols[:, :, :, i, j, :]
        return dx_pad[:, pad:pad + h, pad:pad + w, :] if pad else dx_pad

    class Conv2DNumPy:
        def __init__(self, in_channels, out_channels, kernel_size=3, padding=1, seed=0):
            rng = np.random.default_rng(seed)
            scale = np.sqrt(2.0 / (kernel_size * kernel_size * in_channels))
            self.W = (rng.normal(0, scale, (kernel_size, kernel_size, in_channels, out_channels))).astype("float32")
            self.b = np.zeros(out_channels, dtype="float32")
            self.padding = padding
            self.kernel_size = kernel_size
        def forward(self, x):
            self.x = x
            self.cols, self.x_pad_shape, out_h, out_w = im2col_nhwc(x, self.kernel_size, self.kernel_size, self.padding)
            out = self.cols @ self.W.reshape(-1, self.W.shape[-1]) + self.b
            return out.reshape(x.shape[0], out_h, out_w, self.W.shape[-1])
        def backward(self, dout):
            dout2 = dout.reshape(-1, dout.shape[-1])
            self.dW = (self.cols.T @ dout2).reshape(self.W.shape)
            self.db = dout2.sum(axis=0)
            dcols = dout2 @ self.W.reshape(-1, self.W.shape[-1]).T
            return col2im_nhwc(dcols, self.x.shape, self.x_pad_shape, self.kernel_size, self.kernel_size, self.padding)
        def update(self, lr):
            self.W -= lr * self.dW
            self.b -= lr * self.db

    class ReLUNumPy:
        def forward(self, x):
            self.mask = x > 0
            return x * self.mask
        def backward(self, dout):
            return dout * self.mask
        def update(self, lr):
            pass

    class MaxPool2DNumPy:
        def __init__(self, pool=2, stride=2):
            self.pool = pool
            self.stride = stride
        def forward(self, x):
            self.x = x
            b, h, w, c = x.shape
            out_h, out_w = h // self.stride, w // self.stride
            out = np.empty((b, out_h, out_w, c), dtype=x.dtype)
            self.argmax = np.empty((b, out_h, out_w, c), dtype=np.int32)
            for i in range(out_h):
                for j in range(out_w):
                    window = x[:, i*self.stride:i*self.stride+self.pool, j*self.stride:j*self.stride+self.pool, :]
                    flat = window.reshape(b, self.pool * self.pool, c)
                    self.argmax[:, i, j, :] = flat.argmax(axis=1)
                    out[:, i, j, :] = flat.max(axis=1)
            return out
        def backward(self, dout):
            dx = np.zeros_like(self.x)
            b, out_h, out_w, c = dout.shape
            for i in range(out_h):
                for j in range(out_w):
                    for n in range(b):
                        for ch in range(c):
                            pos = self.argmax[n, i, j, ch]
                            dx[n, i*self.stride + pos // self.pool, j*self.stride + pos % self.pool, ch] += dout[n, i, j, ch]
            return dx
        def update(self, lr):
            pass

    class FlattenNumPy:
        def forward(self, x):
            self.shape = x.shape
            return x.reshape(x.shape[0], -1)
        def backward(self, dout):
            return dout.reshape(self.shape)
        def update(self, lr):
            pass

    class DenseNumPy:
        def __init__(self, in_features, out_features, seed=0):
            rng = np.random.default_rng(seed)
            self.W = rng.normal(0, np.sqrt(2.0 / in_features), (in_features, out_features)).astype("float32")
            self.b = np.zeros(out_features, dtype="float32")
        def forward(self, x):
            self.x = x
            return x @ self.W + self.b
        def backward(self, dout):
            self.dW = self.x.T @ dout
            self.db = dout.sum(axis=0)
            return dout @ self.W.T
        def update(self, lr):
            self.W -= lr * self.dW
            self.b -= lr * self.db

    class SoftmaxCrossEntropyNumPy:
        def forward(self, logits, labels):
            shifted = logits - logits.max(axis=1, keepdims=True)
            exp = np.exp(shifted)
            self.probs = exp / exp.sum(axis=1, keepdims=True)
            self.labels = labels
            return -np.log(self.probs[np.arange(len(labels)), labels] + 1e-9).mean()
        def backward(self):
            grad = self.probs.copy()
            grad[np.arange(len(self.labels)), self.labels] -= 1
            return grad / len(self.labels)

    class SGDNumPy:
        def __init__(self, lr):
            self.lr = lr
        def step(self, layers):
            for layer in layers:
                layer.update(self.lr)

    print("NumPy CNN components defined:", 8)


**Nhận xét kết quả.**

- Các class NumPy đã có `forward()` và `backward()` thật cho Conv2D, ReLU, MaxPool2D, Flatten, Dense và SoftmaxCrossEntropy.
- `SGDNumPy` thực hiện bước update tham số bằng gradient đã tính thủ công, không dùng TensorFlow/PyTorch autograd.
- Conv2D dùng im2col/col2im để giữ code dễ kiểm tra hơn so với nhiều vòng lặp pixel lồng nhau.

# 9. NumPy CNN Training


Test từng NumPy layer, tạo CNN nhỏ, train thật bằng forward/loss/backward/update trên stratified subset và lưu best state.

In [ ]:
if RUN_TRAINING:
    def macro_metrics(y_true, y_pred):
        return {
            "Accuracy": accuracy_score(y_true, y_pred),
            "Macro-P": precision_score(y_true, y_pred, average="macro", zero_division=0),
            "Macro-R": recall_score(y_true, y_pred, average="macro", zero_division=0),
            "Macro-F1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        }

    class SimpleNumPyCNN:
        def __init__(self, seed=SEED):
            self.layers = [
                Conv2DNumPy(1, 2, 3, 1, seed + 1), ReLUNumPy(), MaxPool2DNumPy(),
                Conv2DNumPy(2, 4, 3, 1, seed + 2), ReLUNumPy(), MaxPool2DNumPy(),
                FlattenNumPy(), DenseNumPy(7 * 7 * 4, 16, seed + 3), ReLUNumPy(), DenseNumPy(16, 10, seed + 4),
            ]
            self.loss_fn = SoftmaxCrossEntropyNumPy()
        def forward(self, x, collect_shapes=False):
            shapes = [("Input", x.shape)]
            out = x
            for layer in self.layers:
                out = layer.forward(out)
                if collect_shapes:
                    shapes.append((layer.__class__.__name__, out.shape))
            return (out, shapes) if collect_shapes else out
        def backward(self, grad):
            for layer in reversed(self.layers):
                grad = layer.backward(grad)
        def update(self, lr):
            SGDNumPy(lr).step(self.layers)
        def train_batch(self, x, y, lr):
            logits = self.forward(x)
            loss = self.loss_fn.forward(logits, y)
            grad = self.loss_fn.backward()
            self.backward(grad)
            self.update(lr)
            return float(loss)
        def predict_proba(self, x, batch_size=256):
            probs = []
            for start in range(0, len(x), batch_size):
                logits = self.forward(x[start:start + batch_size])
                shifted = logits - logits.max(axis=1, keepdims=True)
                exp = np.exp(shifted)
                probs.append(exp / exp.sum(axis=1, keepdims=True))
            return np.vstack(probs)
        def state_dict(self):
            state = {}
            for i, layer in enumerate(self.layers):
                if hasattr(layer, "W"):
                    state[f"{i}_W"] = layer.W.copy()
                    state[f"{i}_b"] = layer.b.copy()
            return state
        def load_state_dict(self, state):
            for i, layer in enumerate(self.layers):
                if hasattr(layer, "W"):
                    layer.W[...] = state[f"{i}_W"]
                    layer.b[...] = state[f"{i}_b"]
        def parameter_count(self):
            return int(sum(layer.W.size + layer.b.size for layer in self.layers if hasattr(layer, "W")))

    component_tests = []
    rng_test = np.random.default_rng(SEED + 100)
    unit_x = rng_test.normal(size=(2, 5, 5, 1)).astype("float32")
    unit_grad = rng_test.normal(size=(2, 5, 5, 2)).astype("float32")

    conv_unit = Conv2DNumPy(1, 2, 3, 1, seed=SEED + 101)
    conv_out = conv_unit.forward(unit_x)
    conv_dx = conv_unit.backward(unit_grad)
    component_tests.extend([
        {"Check": "Conv2D output shape", "Value": conv_out.shape},
        {"Check": "Conv2D input grad shape", "Value": conv_dx.shape},
        {"Check": "Conv2D parameter shapes", "Value": (conv_unit.W.shape, conv_unit.b.shape)},
        {"Check": "Conv2D finite output/grad", "Value": bool(np.isfinite(conv_out).all() and np.isfinite(conv_dx).all())},
    ])

    old_weight = float(conv_unit.W[0, 0, 0, 0])
    eps = 1e-3
    conv_unit.W[0, 0, 0, 0] = old_weight + eps
    plus = float((conv_unit.forward(unit_x) * unit_grad).sum())
    conv_unit.W[0, 0, 0, 0] = old_weight - eps
    minus = float((conv_unit.forward(unit_x) * unit_grad).sum())
    conv_unit.W[0, 0, 0, 0] = old_weight
    conv_unit.forward(unit_x)
    conv_unit.backward(unit_grad)
    conv_numeric_grad = (plus - minus) / (2 * eps)
    component_tests.append({"Check": "Conv2D numerical grad close", "Value": bool(np.isclose(conv_numeric_grad, conv_unit.dW[0, 0, 0, 0], atol=2e-2))})

    relu_unit = ReLUNumPy()
    relu_out = relu_unit.forward(conv_out)
    relu_dx = relu_unit.backward(np.ones_like(relu_out))
    component_tests.extend([
        {"Check": "ReLU output shape", "Value": relu_out.shape},
        {"Check": "ReLU grad shape", "Value": relu_dx.shape},
    ])

    pool_unit = MaxPool2DNumPy()
    pool_out = pool_unit.forward(relu_out)
    pool_dx = pool_unit.backward(np.ones_like(pool_out))
    component_tests.extend([
        {"Check": "MaxPool output shape", "Value": pool_out.shape},
        {"Check": "MaxPool grad shape", "Value": pool_dx.shape},
    ])

    flat_unit = FlattenNumPy()
    flat_out = flat_unit.forward(pool_out)
    flat_dx = flat_unit.backward(np.ones_like(flat_out))
    component_tests.extend([
        {"Check": "Flatten output shape", "Value": flat_out.shape},
        {"Check": "Flatten grad shape", "Value": flat_dx.shape},
    ])

    dense_unit = DenseNumPy(flat_out.shape[1], 10, seed=SEED + 102)
    dense_out = dense_unit.forward(flat_out)
    dense_dx = dense_unit.backward(np.ones_like(dense_out) / dense_out.size)
    component_tests.extend([
        {"Check": "Dense output shape", "Value": dense_out.shape},
        {"Check": "Dense grad/input shape", "Value": dense_dx.shape},
        {"Check": "Dense parameter shapes", "Value": (dense_unit.W.shape, dense_unit.b.shape)},
    ])

    loss_unit = SoftmaxCrossEntropyNumPy()
    loss_value = loss_unit.forward(dense_out, np.array([0, 1], dtype="int64"))
    loss_grad = loss_unit.backward()
    component_tests.extend([
        {"Check": "SoftmaxCE loss finite", "Value": bool(np.isfinite(loss_value))},
        {"Check": "Softmax sums approx 1", "Value": bool(np.allclose(loss_unit.probs.sum(axis=1), 1.0, atol=1e-6))},
        {"Check": "Loss grad logits shape", "Value": loss_grad.shape},
    ])

    layer_tests = component_tests.copy()
    test_x = X_train_keras[:4].astype("float32")
    test_y = y_train[:4].astype("int64")
    model_np = SimpleNumPyCNN()
    logits, shape_trace = model_np.forward(test_x, collect_shapes=True)
    loss = model_np.loss_fn.forward(logits, test_y)
    model_np.backward(model_np.loss_fn.backward())
    layer_tests.append({"Check": "Logits shape", "Value": logits.shape})
    layer_tests.append({"Check": "Loss finite", "Value": bool(np.isfinite(loss))})
    layer_tests.append({"Check": "Softmax row sum", "Value": float(model_np.loss_fn.probs[0].sum())})
    for idx, layer in enumerate(model_np.layers):
        if hasattr(layer, "dW"):
            layer_tests.append({"Check": f"Layer {idx} grad W shape", "Value": layer.dW.shape})

    for name, value in model_np.state_dict().items():
        layer_tests.append({"Check": f"Parameter {name} shape", "Value": value.shape})

    tiny_model = SimpleNumPyCNN(seed=SEED + 200)
    tiny_x = X_train_keras[:32].astype("float32")
    tiny_y = y_train[:32].astype("int64")
    initial_tiny_loss = tiny_model.loss_fn.forward(tiny_model.forward(tiny_x), tiny_y)
    for _ in range(10):
        tiny_model.train_batch(tiny_x, tiny_y, 0.02)
    final_tiny_loss = tiny_model.loss_fn.forward(tiny_model.forward(tiny_x), tiny_y)
    layer_tests.append({"Check": "Tiny batch initial loss", "Value": float(initial_tiny_loss)})
    layer_tests.append({"Check": "Tiny batch final loss", "Value": float(final_tiny_loss)})
    layer_tests.append({"Check": "Tiny batch loss decreases", "Value": bool(final_tiny_loss <= initial_tiny_loss)})

    dense = DenseNumPy(3, 2, seed=1)
    tiny_x = np.array([[0.2, -0.1, 0.4]], dtype="float32")
    tiny_grad = np.array([[0.5, -0.2]], dtype="float32")
    out = dense.forward(tiny_x)
    dense.backward(tiny_grad)
    eps = 1e-4
    old = dense.W[0, 0]
    dense.W[0, 0] = old + eps
    plus = (dense.forward(tiny_x) * tiny_grad).sum()
    dense.W[0, 0] = old - eps
    minus = (dense.forward(tiny_x) * tiny_grad).sum()
    dense.W[0, 0] = old
    numeric_grad = (plus - minus) / (2 * eps)
    layer_tests.append({"Check": "Dense numerical grad close", "Value": bool(np.isclose(numeric_grad, dense.dW[0, 0], atol=1e-3))})

    shape_trace_df = pd.DataFrame(shape_trace, columns=["Stage", "Shape"])
    display(pd.DataFrame(layer_tests))
    display(shape_trace_df)

    numpy_train_size = 6000
    numpy_valid_size = 1200
    idx_train_np, _ = train_test_split(
        np.arange(len(X_train_keras)), train_size=numpy_train_size, random_state=SEED, stratify=y_train
    )
    idx_valid_np, _ = train_test_split(
        np.arange(len(X_valid_keras)), train_size=numpy_valid_size, random_state=SEED, stratify=y_valid
    )
    X_np_train = X_train_keras[idx_train_np].astype("float32")
    y_np_train = y_train[idx_train_np].astype("int64")
    X_np_valid = X_valid_keras[idx_valid_np].astype("float32")
    y_np_valid = y_valid[idx_valid_np].astype("int64")

    batch_size_np = 128
    epochs_np = 3
    lr_np = 0.04
    history_np = []
    best_np = {"Macro-F1": -1, "state": None, "epoch": 0}
    rng = np.random.default_rng(SEED)
    start_time = time.perf_counter()

    for epoch in range(1, epochs_np + 1):
        order = rng.permutation(len(X_np_train))
        losses = []
        for start in range(0, len(order), batch_size_np):
            batch_ids = order[start:start + batch_size_np]
            loss_value = model_np.train_batch(X_np_train[batch_ids], y_np_train[batch_ids], lr_np)
            losses.append(loss_value)
        val_prob = model_np.predict_proba(X_np_valid)
        val_pred = val_prob.argmax(axis=1)
        metrics = macro_metrics(y_np_valid, val_pred)
        row = {"Epoch": epoch, "TrainLoss": float(np.mean(losses)), **metrics}
        history_np.append(row)
        if metrics["Macro-F1"] > best_np["Macro-F1"]:
            best_np = {"Macro-F1": metrics["Macro-F1"], "state": model_np.state_dict(), "epoch": epoch}

    numpy_train_time = time.perf_counter() - start_time
    model_np.load_state_dict(best_np["state"])
    np.savez(MODEL_DIR / "numpy_cnn_best_state.npz", **best_np["state"])

    numpy_history = pd.DataFrame(history_np)
    numpy_history.to_csv(METRIC_DIR / "numpy_cnn_history.csv", index=False)
    numpy_training_summary = {
        "Train samples": numpy_train_size,
        "Validation samples": numpy_valid_size,
        "Epochs": epochs_np,
        "Batch size": batch_size_np,
        "Optimizer": "SGD",
        "Learning rate": lr_np,
        "Best epoch": best_np["epoch"],
        "Train time": numpy_train_time,
        "Parameters": model_np.parameter_count(),
    }
    display(pd.DataFrame([numpy_training_summary]).T.rename(columns={0: "Value"}))
    display(numpy_history.round(4))


**Nhận xét kết quả.**

- Layer tests kiểm tra output shape, gradient shape, finite values, parameter shape và softmax row sums trước khi train.
- Numerical gradient check trên Conv2D đối chiếu gradient backward với sai phân hữu hạn cho một weight nhỏ.
- Tiny-batch test xác nhận loss giảm sau vài bước SGD, rồi NumPy CNN train thật trên 6,000 samples và lưu best state theo validation Macro-F1.

# 10. Tensor Shape and Parameter Inspection


Tính parameter từng layer của NumPy CNN và đối chiếu shape flow qua mạng.

In [ ]:
if RUN_TRAINING:
    param_rows = []
    for idx, layer in enumerate(model_np.layers):
        if hasattr(layer, "W"):
            param_rows.append({
                "LayerIndex": idx,
                "Layer": layer.__class__.__name__,
                "WeightShape": layer.W.shape,
                "BiasShape": layer.b.shape,
                "Parameters": int(layer.W.size + layer.b.size),
            })
    param_table = pd.DataFrame(param_rows)
    total_params_np = int(param_table["Parameters"].sum())
    display(param_table)
    print("Total NumPy CNN parameters:", total_params_np)

    manual_conv1 = (3 * 3 * 1 * 2) + 2
    print("Manual Conv1 parameters:", manual_conv1)


**Nhận xét kết quả.**

- Conv1 có `(3 x 3 x 1 x 2) + 2` parameters: 18 weights và 2 bias.
- Tổng parameters nhỏ để NumPy training khả thi, nhưng vẫn có Conv, ReLU, Pool, Flatten và Dense đúng pipeline CNN.

# 11. CNN with Keras/TensorFlow


Xây Keras Basic CNN tương đương về ý tưởng với NumPy CNN nhưng dùng full framework để train hiệu quả hơn.

In [ ]:
if RUN_TRAINING:
    def save_json(path, payload):
        with Path(path).open('w', encoding='utf-8') as f:
            json.dump(payload, f, ensure_ascii=False, indent=2)

    def evaluate_predictions(y_true, y_pred):
        return {'Accuracy': accuracy_score(y_true, y_pred), 'Macro-P': precision_score(y_true, y_pred, average='macro', zero_division=0), 'Macro-R': recall_score(y_true, y_pred, average='macro', zero_division=0), 'Macro-F1': f1_score(y_true, y_pred, average='macro', zero_division=0)}


**Nhận xét kết quả.**

- `model.summary()` cho thấy spatial size giảm sau mỗi MaxPool: 28x28 → 14x14 → 7x7.
- Channel tăng từ 1 lên 8 rồi 16 nhờ Conv2D filters.
- Output cuối là 10 probabilities cho 10 digit classes.

# 12. Keras Training


Train Keras Basic CNN với validation, checkpoint, EarlyStopping và lưu training curves.

In [ ]:
if RUN_TRAINING:
    BATCH_SIZE = 256
    MAX_EPOCHS = 5


**Nhận xét kết quả.**

- Keras evaluation dùng test split độc lập của MNIST và đạt Accuracy 0.9820, Macro-F1 0.9819.
- Macro metrics đánh giá trung bình giữa 10 chữ số, còn confusion matrix cho biết digit nào bị nhầm với digit nào.

# 13. Keras Evaluation


Load Keras checkpoint tốt nhất, evaluate trên test set và lưu metrics/confusion matrix.

**Nhận xét kết quả.**

- Keras evaluation dùng test split độc lập của MNIST.
- Macro metrics đánh giá trung bình giữa 10 chữ số, còn confusion matrix cho biết digit nào bị nhầm với digit nào.

# 14. CNN with PyTorch


Xây PyTorch CNN tương đương về ý tưởng với Keras Basic CNN và kiểm tra input/output shape trước khi train.

**Nhận xét kết quả.**

- PyTorch nhận ảnh theo layout `(B, C, H, W)`.
- Output logits có shape `(B, 10)`, nghĩa là mỗi ảnh có 10 score chưa softmax.

# 15. PyTorch Training


Train PyTorch CNN bằng loop thủ công có zero grad, forward, loss, backward và optimizer step.

In [ ]:
if RUN_TRAINING:
    best_torch = {'Macro-F1': -1, 'epoch': 0}
    torch_history = []


**Nhận xét kết quả.**

- Training loop hiển thị rõ các bước gradient thủ công của PyTorch.
- Best checkpoint được chọn theo validation Macro-F1 để cân bằng chất lượng giữa các class.

# 16. PyTorch Evaluation


Load PyTorch checkpoint tốt nhất và evaluate trên test set bằng cùng protocol metrics với Keras.

**Nhận xét kết quả.**

- Bảng so sánh cùng các cột Accuracy, Macro-P, Macro-R, Macro-F1, Params, Train Samples, Best Epoch, Train Time và Prediction Time.
- NumPy CNN đạt Macro-F1 0.7733 với subset nhỏ; Keras đạt 0.9819 và PyTorch đạt 0.9766 trên full training split.
- Không nên kết luận framework nào luôn tốt hơn từ một run; runtime phụ thuộc hardware/backend và mức tối ưu của implementation.

# 17. Framework Comparison


Evaluate NumPy CNN trên test set và so sánh NumPy, Keras, PyTorch bằng cùng metric classification.

In [ ]:
if RUN_TRAINING:
    start = time.perf_counter()
    numpy_test_prob = model_np.predict_proba(X_test_keras.astype("float32"), batch_size=256)
    numpy_pred = numpy_test_prob.argmax(axis=1)
    numpy_prediction_time = time.perf_counter() - start
    numpy_metrics = macro_metrics(y_test, numpy_pred)
    numpy_cm = confusion_matrix(y_test, numpy_pred)
    numpy_result = {
        "Model": "NumPy CNN",
        **numpy_metrics,
        "Params": int(model_np.parameter_count()),
        "Train Samples": int(numpy_train_size),
        "Best Epoch": int(best_np["epoch"]),
        "Train Time": float(numpy_train_time),
        "Prediction Time": float(numpy_prediction_time),
    }
    save_json(METRIC_DIR / "numpy_cnn_test_metrics.json", numpy_result)

    fig, ax = plt.subplots(figsize=(5.5, 5))
    im = ax.imshow(numpy_cm, cmap="Purples")
    ax.set_title("NumPy CNN Confusion Matrix")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_xticks(range(10))
    ax.set_yticks(range(10))
    fig.colorbar(im, ax=ax, fraction=0.046)
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / "numpy_confusion_matrix.png", dpi=160)
    plt.show()

    framework_comparison = pd.DataFrame([numpy_result, keras_result, torch_result])
    framework_comparison.to_csv(METRIC_DIR / "framework_comparison.csv", index=False)
    display(framework_comparison.round(4))


**Nhận xét kết quả.**

- Bảng so sánh cùng các cột Accuracy, Macro-P, Macro-R, Macro-F1, Params, Train Samples, Best Epoch, Train Time và Prediction Time.
- NumPy CNN đạt Macro-F1 0.7733 với subset nhỏ; Keras đạt 0.9819 và PyTorch đạt 0.9766 trên full training split.
- Không nên kết luận framework nào luôn tốt hơn từ một run; runtime phụ thuộc hardware/backend và mức tối ưu của implementation.

# 18. CNN Architecture Improvements


Thử cải tiến nhẹ trên MNIST: M1 Basic CNN và M2 Basic CNN + BatchNorm + Dropout.

**Nhận xét kết quả.**

- Bảng test metrics cho biết kết quả cuối của từng model sau khi đã train.
- `selection_results` dùng validation Macro-F1 để chọn final model, nên test set không bị dùng để tune/chọn model.
- MNIST dễ nên các framework CNN đạt kết quả cao; NumPy CNN nhỏ có mục tiêu giáo dục hơn là tối ưu performance.

# 19. Improved Model Comparison


Tổng hợp Basic CNN, improved CNN và chọn candidate tốt nhất theo Macro-F1.

**Nhận xét kết quả.**

- Bảng này chọn model tốt nhất theo Macro-F1 sau khi có NumPy, Keras, PyTorch và improved Keras.
- Model tốt nhất trong run này là `M2 Basic CNN + BatchNorm + Dropout` với Accuracy 0.9851 và Macro-F1 0.9850.
- MNIST dễ nên các framework CNN đạt kết quả cao; NumPy CNN nhỏ có mục tiêu giáo dục hơn là tối ưu performance.

# 20. Feature Map Inspection


Lấy ảnh MNIST thật và hiển thị feature maps từ block Conv đầu và block Conv thứ hai của Keras model tốt nhất.

**Nhận xét kết quả.**

- Feature maps là activation thật từ Conv2D layers trên ảnh MNIST.
- Notebook không gán cứng một channel là edge detector; output chỉ cho thấy mỗi filter phản ứng khác nhau với nét chữ.

# 21. Error Analysis


Hiển thị các dự đoán sai có confidence cao nhất và dùng confusion matrix để xem digit nào dễ nhầm.

In [ ]:
if RUN_TRAINING:
    confusion_pairs = []


**Nhận xét kết quả.**

- Error examples hiển thị ảnh thật, true label, predicted label và confidence.
- Confusion pair lớn nhất trong run này là true `4` bị dự đoán thành `9` với 13 cases.
- Các lỗi này thường liên quan đến nét viết mơ hồ hoặc chữ số có hình dáng gần nhau.

# 22. Final Model


Chọn model cuối theo Macro-F1, lưu final metrics, final model reference và confusion matrix cuối.

**Nhận xét kết quả.**

- Final model được chọn bằng validation Macro-F1, sau đó mới đọc test metrics tương ứng để báo cáo.
- Model cuối được lưu riêng để dễ tìm khi viết báo cáo hoặc demo.
- Metrics cuối là kết quả thật từ model đã chọn, không phải số điền trước.

# 23. Final Results


Lưu benchmark, metadata và liệt kê toàn bộ artifacts MNIST CNN đã tạo.

In [ ]:
if RUN_TRAINING:
    artifact_listing = []


**Nhận xét kết quả.**

- Bảng benchmark cuối tổng hợp kết quả thật của NumPy CNN, Keras CNN, PyTorch CNN và improved CNN.
- Artifact listing xác nhận notebook đã lưu 5 model files, 20 metric files và 13 figures dưới `pipeline04/mnist/`.
- Tổng recorded train time hiện tại khoảng 58.33 giây.

In [ ]:
from IPython.display import display
import pandas as pd
_saved = _repo / 'A04/artifacts/pipeline04/mnist/metrics'
for _file in sorted(_saved.glob('*comparison*.csv')):
    print(_file.name)
    display(pd.read_csv(_file))
